# Comparison: `stata_type=False` (sklearn) vs `stata_type=True` (Stata-faithful)

This notebook demonstrates that both estimation backends produce virtually identical results.

- **`stata_type=False`** (default): uses scikit-learn `LinearRegression` and `LogisticRegression`
- **`stata_type=True`**: uses `statsmodels` OLS and a custom Newton-Raphson logit that mimics Stata

Specification: `order=1, aoss_vs_waoss=True, controls=["lngpinc"]`

In [ ]:
import sys, os, time, warnings
import pandas as pd
import numpy as np

sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))
from did_multiplegt_stat import did_multiplegt_stat, summary_did_multiplegt_stat

warnings.filterwarnings("ignore")

## 1. Load data

In [ ]:
df = pd.read_stata("gazoline_did_multiplegt_stat.dta")
print(f"Observations: {len(df):,}")
print(f"Units: {df['id'].nunique()}")
print(f"Periods: {df['year'].nunique()}")
df.head()

## 2. Run with `stata_type=False` (sklearn, default)

In [ ]:
t0 = time.time()
res_sklearn = did_multiplegt_stat(
    df, Y="lngca", ID="id", Time="year", D="tau",
    order=1,
    aoss_vs_waoss=True,
    controls=["lngpinc"],
    stata_type=False,
)
t_sklearn = time.time() - t0
print(f"\nElapsed: {t_sklearn:.1f}s")
summary_did_multiplegt_stat(res_sklearn)

## 3. Run with `stata_type=True` (Stata-faithful)

In [ ]:
t0 = time.time()
res_stata = did_multiplegt_stat(
    df, Y="lngca", ID="id", Time="year", D="tau",
    order=1,
    aoss_vs_waoss=True,
    controls=["lngpinc"],
    stata_type=True,
)
t_stata = time.time() - t0
print(f"\nElapsed: {t_stata:.1f}s")
summary_did_multiplegt_stat(res_stata)

## 4. Side-by-side comparison

In [ ]:
tbl_sk = res_sklearn["results"]["table"]
tbl_st = res_stata["results"]["table"]

rows = []
for idx in tbl_sk.index:
    for col in ["Estimate", "SE", "LB CI", "UB CI"]:
        v_sk = tbl_sk.loc[idx, col]
        v_st = tbl_st.loc[idx, col]
        if v_st != 0:
            pct_diff = abs(v_sk - v_st) / abs(v_st) * 100
        else:
            pct_diff = 0.0 if v_sk == 0 else np.inf
        rows.append({
            "Metric": idx,
            "Stat": col,
            "sklearn": v_sk,
            "Stata-type": v_st,
            "Diff": v_sk - v_st,
            "% Diff": pct_diff,
        })

comp = pd.DataFrame(rows)
comp["% Diff"] = comp["% Diff"].map(lambda x: f"{x:.6f}%")
comp

## 5. AOSS vs WAOSS test comparison

In [ ]:
avw_sk = res_sklearn["results"]["aoss_vs_waoss"]
avw_st = res_stata["results"]["aoss_vs_waoss"]

rows_avw = []
for idx in avw_sk.index:
    for col in avw_sk.columns:
        v_sk = avw_sk.loc[idx, col]
        v_st = avw_st.loc[idx, col]
        if pd.notna(v_st) and v_st != 0:
            pct_diff = abs(v_sk - v_st) / abs(v_st) * 100
        else:
            pct_diff = 0.0 if (pd.isna(v_sk) and pd.isna(v_st)) or v_sk == 0 else np.inf
        rows_avw.append({
            "Metric": idx,
            "Stat": col,
            "sklearn": v_sk,
            "Stata-type": v_st,
            "Diff": v_sk - v_st if pd.notna(v_sk) and pd.notna(v_st) else np.nan,
            "% Diff": f"{pct_diff:.6f}%",
        })

comp_avw = pd.DataFrame(rows_avw)
comp_avw

## 6. Summary

In [ ]:
# Extract numeric % diffs for summary
pct_vals = comp["% Diff"].str.rstrip("%").astype(float)
max_diff = pct_vals.max()

print("=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"Max % difference (main table):  {max_diff:.6f}%")
print(f"Mean % difference (main table): {pct_vals.mean():.6f}%")
print(f"")
print(f"Time sklearn:    {t_sklearn:.1f}s")
print(f"Time Stata-type: {t_stata:.1f}s")
print(f"Speedup:         {t_stata/t_sklearn:.2f}x" if t_sklearn > 0 else "")
print()
if max_diff < 0.1:
    print("CONCLUSION: Both backends produce practically identical results.")
    print("Differences are due to floating-point precision (Newton-Raphson vs LBFGS).")
else:
    print(f"NOTE: Max difference is {max_diff:.4f}% -- check controls path logit precision.")